# Cube Nano - SegFormer-B0 training on 95-Cloud

This Colab notebook trains the RGB, two-class SegFormer-B0 cloud segmenter using the repository pipeline rather than a duplicate notebook-only training loop. It preserves native scene dimensions during training and validation with batch size 1, then exports the fixed runtime graph with input `[1, 3, 256, 256]`.

Before running, select a GPU runtime and add the `KAGGLE_API_TOKEN` secret in Colab. Set `repo_ref` to an immutable commit that contains the SegFormer files before a reproducible run.

> The default mode is `research_baseline`. It creates a checkpoint and evidence bundle, but does not mark a deployment release as valid. When enabled in `CFG`, the notebook downloads the `nvidia/mit-b0` pretrained encoder, records its checksum, and leaves the decoder/release approval gates separate.


## 1. Install the Colab dependencies

Colab supplies the CUDA-enabled PyTorch build. This cell deliberately does not install the CPU lockfile, which would replace that build.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

required_packages = {
    'tifffile': 'tifffile',
    'tqdm': 'tqdm',
    'yaml': 'PyYAML',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'kaggle': 'kaggle',
    'pytest': 'pytest',
    'transformers': 'transformers',
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Training will stop later unless a GPU runtime is selected.')


## 2. Configure the run and source revision

Use a commit SHA for `repo_ref` when reproducing a candidate. `raw_audit` is intentionally explicit: a release-candidate run stops when the required radiometric fields are still marked `UNVERIFIED`.


In [ ]:
import datetime as dt
import hashlib
import json
import os
import platform
import shlex
import shutil
from pathlib import Path

CFG = {
    'repo_url': 'https://github.com/hoxuanphu/cube_nano.git',
    'repo_ref': 'main',
    'kaggle_slug': 'sorour/95cloud-cloud-segmentation-on-satellite-images',
    'data_root_override': None,
    'drive_cube_nano_path': '/content/drive/MyDrive/cube_nano',  # Add the shared folder as a My Drive shortcut in the secondary account.
    'raw_on_drive': True,
    'processed_on_drive': False,  # Active train/val/test data stays on fast local Colab storage.
    'processed_cache_name': 'segformer_rgb_native_v2',
    'processed_cache_archive': True,  # Persist one sequential archive on Drive, not thousands of NPY files.
    'processed_cache_reserve_gib': 8.0,  # Keep headroom for the runtime, checkpoints, and reports.
    'persist_raw_masks': False,  # The training/evaluation contract only consumes mask + validity.
    'preprocess_workers': 2,  # Small bounded read parallelism for Drive-backed TIFFs.
    'reuse_raw_audit_cache': True,
    'reuse_processed_cache': True,
    'remove_kaggle_archive': True,
    'cleanup_stale_local_workspace': False,
    'cleanup_local_after_bundle': True,
    'run_mode': 'research_baseline',  # research_baseline or release_candidate
    'rebuild_processed_data': False,
    'move_split': False,  # Retained for config compatibility; direct preprocessing now assigns splits.
    'run_regression_tests': True,
    'seed': 42,
    'cloud_ratio_threshold': 0.10,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    'epochs': 50,
    'learning_rate': 6e-5,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    'early_stopping_patience': 12,
    'use_amp': True,
    'train_batch_size': 1,  # Change this value to 2, 4, 8, ... as GPU memory allows.
    'train_preserve_native_size': True,  # False selects the 256x256 crop/tile pipeline.
    'use_pretrained_segformer': True,
    'pretrained_segformer_model_id': 'nvidia/mit-b0',
    'max_false_clear_rate': 0.05,
    'threshold_start_bp': 1000,
    'threshold_stop_bp': 10000,
    'threshold_step_bp': 100,
    'bootstrap_samples': 1000,
}

raw_audit = {
    'sensor_id': 'UNVERIFIED',
    'platform_id': 'UNVERIFIED',
    'product_type': 'UNVERIFIED',
    'processing_level': 'UNVERIFIED',
    'units': 'UNVERIFIED',
    'scale_offset': 'UNVERIFIED',
    'nodata': 'UNVERIFIED',
    'saturation': 'UNVERIFIED',
    'gsd': 'UNVERIFIED',
    'band_order': ['red', 'green', 'blue'],
    'ground_truth_encoding_confirmed': False,
    'ground_truth_clear_values': [0],
    'ground_truth_cloud_values': [1, 255],
    'invalid_ground_truth_values': [],
}

if CFG['run_mode'] not in {'research_baseline', 'release_candidate'}:
    raise ValueError('run_mode must be research_baseline or release_candidate')
if not 0 <= CFG['max_false_clear_rate'] <= 1:
    raise ValueError('max_false_clear_rate must be in [0, 1]')
if not 0 <= CFG['val_ratio'] < 1 or not 0 <= CFG['test_ratio'] < 1:
    raise ValueError('split ratios must be in [0, 1)')
if CFG['val_ratio'] + CFG['test_ratio'] >= 1:
    raise ValueError('validation and test ratios must leave a train split')

CONTENT = Path('/content')
PROJECT = CONTENT / 'cube_nano'
RAW = CONTENT / '95cloud_kaggle'
RUN = CONTENT / 'segformer_95cloud_run'
PROCESSED = RUN / 'data' / 'processed'
PROCESSED_ALL = PROCESSED / 'all'
CHECKPOINTS = RUN / 'checkpoints'
RESULTS = RUN / 'results'
CONTRACTS = RUN / 'contracts'
ARTIFACTS = RUN / 'artifacts'
DELIVERABLES = RUN / 'deliverables'
for directory in (RUN, CHECKPOINTS, RESULTS, CONTRACTS, ARTIFACTS, DELIVERABLES):
    directory.mkdir(parents=True, exist_ok=True)

print(json.dumps(CFG, indent=2, sort_keys=True))
print('Run directory:', RUN)


## 3. Mount Google Drive for persistent artifacts and caches

This copy is for a `cube_nano` folder shared from another Google account. In the secondary account, open the shared folder and add a shortcut to My Drive, or change `CFG['drive_cube_nano_path']` to its mounted path. The training workspace remains on `/content`; raw TIFFs stay in the shared folder while active processed data stays local. A processed cache archive, checkpoints, reports, and the final evidence bundle are persisted in the shared folder.


In [ ]:
from google.colab import drive
import filecmp

DRIVE_MOUNT = '/content/drive'
drive.mount(DRIVE_MOUNT, force_remount=False)
DRIVE_CUBE_NANO = Path(CFG['drive_cube_nano_path']).expanduser().resolve()
if not DRIVE_CUBE_NANO.is_dir():
    raise FileNotFoundError(
        f'Shared cube_nano folder is not mounted: {DRIVE_CUBE_NANO}. '
        'In the secondary Google account, add the shared folder as a shortcut to My Drive, '
        "then rerun this cell, or set CFG['drive_cube_nano_path'] to the mounted path."
    )
DRIVE_ROOT = DRIVE_CUBE_NANO / 'segformer_95cloud'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_DELIVERABLES = DRIVE_ROOT / 'deliverables'
DRIVE_CACHE = DRIVE_ROOT / 'cache'
DRIVE_AUDIT_CACHE = DRIVE_CACHE / 'raw_dataset_audit.json'
DRIVE_PROCESSED_ROOT = DRIVE_ROOT / 'processed'
DRIVE_PROCESSED_ARCHIVE = DRIVE_PROCESSED_ROOT / f"{CFG['processed_cache_name']}.tar"
DRIVE_PROCESSED_METADATA = DRIVE_PROCESSED_ROOT / f"{CFG['processed_cache_name']}.json"
def ensure_drive_directory(directory):
    directory = Path(directory)
    if directory.exists():
        if not directory.is_dir():
            raise NotADirectoryError(f'Drive path exists but is not a directory: {directory}')
        return directory
    directory.mkdir(parents=True)
    return directory

for directory in (DRIVE_CHECKPOINTS, DRIVE_RESULTS, DRIVE_DELIVERABLES, DRIVE_CACHE):
    ensure_drive_directory(directory)

def copy_if_absent_or_same(source, destination):
    source = Path(source)
    destination = Path(destination)
    if not source.is_file():
        raise FileNotFoundError(f'Cannot copy missing file: {source}')
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if not destination.is_file():
            raise FileExistsError(f'Destination exists and is not a file: {destination}')
        if source.stat().st_size == destination.stat().st_size and filecmp.cmp(source, destination, shallow=False):
            print('Reusing existing identical file:', destination)
            return destination
        raise FileExistsError(
            f'Destination already contains different data: {destination}. '
            'Use a new cache/run name instead of overwriting it.'
        )
    shutil.copy2(source, destination)
    return destination

def save_to_drive(source, destination):
    destination = Path(destination)
    ensure_drive_directory(destination.parent)
    return copy_if_absent_or_same(source, destination)

def restore_drive_artifact(destination, *candidates, validator=None):
    destination = Path(destination)
    if destination.is_file():
        return validator(destination) if validator else True
    for candidate in candidates:
        candidate = Path(candidate)
        if not candidate.is_file() or (validator and not validator(candidate)):
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidate, destination)
        print('Restored existing Drive artifact:', destination)
        return True
    return False

def json_cache_matches(path, expected):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8')).get('_cube_nano_cache') == expected
    except (OSError, json.JSONDecodeError):
        return False

def write_checked_text(destination, text, *, replace_stale=False):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if not destination.is_file():
            raise FileExistsError(f'Destination exists and is not a file: {destination}')
        if destination.read_text(encoding='utf-8') == text:
            print('Reusing existing identical metadata:', destination)
            return destination
        if not replace_stale:
            raise FileExistsError(f'Destination already contains different metadata: {destination}')
        print('Updating stale metadata cache in place:', destination)
    destination.write_text(text, encoding='utf-8')
    return destination

LOCAL_RAW = RAW
LOCAL_PROCESSED = PROCESSED
if CFG['raw_on_drive'] and not CFG['data_root_override']:
    RAW = DRIVE_ROOT / 'raw_95cloud'
    ensure_drive_directory(RAW)
    print('Raw dataset will be stored on Drive:', RAW)
else:
    print('Raw dataset location:', RAW)
PROCESSED = LOCAL_PROCESSED
ensure_drive_directory(DRIVE_PROCESSED_ROOT)
print('Active processed data will be kept locally:', PROCESSED)
print('Persistent processed cache archive:', DRIVE_PROCESSED_ARCHIVE)

def disk_report(label):
    usage = shutil.disk_usage(CONTENT)
    gib = 1024 ** 3
    print(f'{label}: {usage.free / gib:.1f} GiB free / {usage.total / gib:.1f} GiB total')

def remove_content_path(path):
    path = Path(path).resolve()
    content_root = CONTENT.resolve()
    if path == content_root or content_root not in path.parents:
        raise ValueError(f'Refusing to remove a path outside the Colab content directory: {path}')
    if path.exists():
        shutil.rmtree(path)
        print('Removed stale local path:', path)

if CFG['cleanup_stale_local_workspace']:
    remove_content_path(LOCAL_RAW)
    remove_content_path(LOCAL_PROCESSED)
else:
    print('Set cleanup_stale_local_workspace=True and rerun this cell to remove a previous failed local run.')
for directory in (RUN, CHECKPOINTS, RESULTS, CONTRACTS, ARTIFACTS, DELIVERABLES):
    directory.mkdir(parents=True, exist_ok=True)
disk_report('After Drive setup')

print('Drive persistence root:', DRIVE_ROOT)


## 4. Clone the exact repository revision

The notebook checks out the requested ref in detached mode and writes the resolved commit to the run provenance. It refuses a non-Git directory at the clone path instead of deleting it.


In [ ]:
def run_command(label, command, *, cwd=None):
    command = [str(item) for item in command]
    print(f'[{label}]')
    print('$', shlex.join(command))
    completed = subprocess.run(command, cwd=str(cwd) if cwd else None, check=False)
    if completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command)

source_provenance_path = RESULTS / 'source_provenance.json'
reuse_local_source = False
if PROJECT.exists() and not (PROJECT / '.git').is_dir():
    raise FileExistsError(f'Clone target is not a Git repository: {PROJECT}')
if PROJECT.exists() and (PROJECT / '.git').is_dir() and source_provenance_path.is_file():
    try:
        previous_source = json.loads(source_provenance_path.read_text(encoding='utf-8'))
        current_source_revision = subprocess.check_output(
            ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True
        ).strip()
        reuse_local_source = (
            previous_source.get('repository_url') == CFG['repo_url']
            and previous_source.get('requested_ref') == CFG['repo_ref']
            and previous_source.get('resolved_commit') == current_source_revision
        )
    except (OSError, json.JSONDecodeError, subprocess.CalledProcessError):
        reuse_local_source = False
if reuse_local_source:
    print('Reusing checked-out repository:', current_source_revision)
else:
    if not PROJECT.exists():
        run_command('Clone repository', ['git', 'clone', '--no-checkout', CFG['repo_url'], PROJECT])
    run_command('Fetch requested ref', ['git', '-C', PROJECT, 'fetch', '--depth', '1', 'origin', CFG['repo_ref']])
    run_command('Checkout requested ref', ['git', '-C', PROJECT, 'checkout', '--detach', 'FETCH_HEAD'])

required_entrypoints = (
    'src/data/preprocess_95cloud.py',
    'src/data/split_dataset.py',
    'src/data/segmentation_dataset.py',
    'src/models/segformer_b0.py',
    'src/train_segmentation.py',
    'src/eval_segmentation.py',
    'src/export_segformer_onnx.py',
    'sat_ai/segformer_model_manifest.yaml',
)
missing_entrypoints = [relative for relative in required_entrypoints if not (PROJECT / relative).is_file()]
if missing_entrypoints:
    raise FileNotFoundError(
        'The selected repo_ref does not contain the SegFormer pipeline: ' + ', '.join(missing_entrypoints)
    )

source_revision = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True
).strip()
source_provenance = {
    'repository_url': CFG['repo_url'],
    'requested_ref': CFG['repo_ref'],
    'resolved_commit': source_revision,
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
}
if not reuse_local_source:
    source_provenance_path.write_text(
        json.dumps(source_provenance, indent=2, sort_keys=True), encoding='utf-8'
    )
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Resolved commit:', source_revision)


## 5. Download and locate 95-Cloud

The Kaggle token is read from Colab Secrets and stays in the process environment only. Set `data_root_override` when the raw TIFF files are already mounted from Drive.


In [ ]:
from google.colab import userdata
from src.data.preprocess_95cloud import discover_scene_files

def has_tiff_files(directory):
    directory = Path(directory)
    return directory.is_dir() and any(
        path.is_file() and path.suffix.lower() in {'.tif', '.tiff'}
        for path in directory.rglob('*')
    )

if CFG['data_root_override']:
    raw_root = Path(CFG['data_root_override']).expanduser().resolve()
    if not raw_root.is_dir():
        raise FileNotFoundError(f'data_root_override does not exist: {raw_root}')
else:
    ensure_drive_directory(RAW)
    if has_tiff_files(RAW):
        try:
            existing_raw_scenes = discover_scene_files(RAW, channels=3)
        except (FileNotFoundError, ValueError) as exc:
            raise RuntimeError(
                f'Raw Drive directory contains TIFF files but is incomplete or duplicated: {RAW}. '
                'Refusing to download again into the same directory.'
            ) from exc
        print(f'Reusing complete raw dataset on Drive: {len(existing_raw_scenes)} scenes')
    else:
        existing_archives = sorted(RAW.glob('*.zip'))
        if existing_archives:
            raise RuntimeError(
                f'Raw Drive directory contains archive(s) but no TIFF files: {existing_archives}. '
                'Refusing a second download; inspect or remove the incomplete archive first.'
            )
        kaggle_token = userdata.get('KAGGLE_API_TOKEN')
        if not kaggle_token:
            raise RuntimeError('Colab Secret KAGGLE_API_TOKEN is missing or empty')
        os.environ['KAGGLE_API_TOKEN'] = kaggle_token
        run_command(
            'Download 95-Cloud from Kaggle',
            ['kaggle', 'datasets', 'download', '-d', CFG['kaggle_slug'], '-p', RAW, '--unzip'],
        )
    if CFG['remove_kaggle_archive']:
        for archive in RAW.glob('*.zip'):
            archive.unlink()
            print('Removed extracted Kaggle archive:', archive)
    raw_root = RAW

def locate_dataset_root(root):
    root = Path(root).resolve()
    tiff_parents = {path.parent for path in root.rglob('*') if path.suffix.lower() in {'.tif', '.tiff'}}
    candidates = [root]
    for parent in sorted(tiff_parents, key=lambda value: (len(value.parts), str(value))):
        candidates.extend([parent, *parent.parents[:5]])
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen or not candidate.is_dir():
            continue
        seen.add(candidate)
        try:
            scenes = discover_scene_files(candidate, channels=3)
        except (FileNotFoundError, ValueError):
            continue
        if scenes:
            return candidate, scenes
    raise FileNotFoundError(f'Could not locate complete RGB plus GT scenes under {root}')

DATA_ROOT, SCENE_FILES = locate_dataset_root(raw_root)
print('Dataset root:', DATA_ROOT)
print('Complete RGB plus GT scenes:', len(SCENE_FILES))
disk_report('After raw dataset resolution')


## 6. Freeze the current contracts and audit the raw pairs

This creates a raw-file manifest, validates RGB/GT shape and dtype consistency, records observed GT values, and writes the declared validity policy. The first run reads each TIFF and computes SHA-256 values; later runs reuse `cache/raw_dataset_audit.json` when the raw-file size/mtime fingerprint and audit policy still match. Set `reuse_raw_audit_cache=False` to force a fresh audit. It does not invent sensor or radiometric metadata: fill `raw_audit` before switching to `release_candidate`.


In [ ]:
import numpy as np
import tifffile
import yaml

segformer_manifest = yaml.safe_load((PROJECT / 'sat_ai/segformer_model_manifest.yaml').read_text(encoding='utf-8'))
acceptance_profile = yaml.safe_load((PROJECT / 'sat_ai/acceptance_profile.yaml').read_text(encoding='utf-8'))
if segformer_manifest.get('model_task') != 'semantic_cloud_segmentation':
    raise ValueError('SegFormer manifest does not declare semantic_cloud_segmentation')
if segformer_manifest['input_spec']['band_order'] != ['red', 'green', 'blue']:
    raise ValueError('The notebook only supports canonical RGB band order')
if segformer_manifest['input_spec']['input_shape'] != [None, 3, 256, 256]:
    raise ValueError('Runtime input contract must remain [null, 3, 256, 256]')
if segformer_manifest['output']['model_output']['shape'] != [1, 2, 64, 64]:
    raise ValueError('Unexpected SegFormer logits contract')
if acceptance_profile['quality']['max_false_clear_rate'] != CFG['max_false_clear_rate']:
    raise ValueError('CFG max_false_clear_rate must match the acceptance profile')

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

declared_clear = {int(value) for value in raw_audit['ground_truth_clear_values']}
declared_cloud = {int(value) for value in raw_audit['ground_truth_cloud_values']}
declared_invalid = {int(value) for value in raw_audit['invalid_ground_truth_values']}
if declared_clear & declared_cloud or declared_clear & declared_invalid or declared_cloud & declared_invalid:
    raise ValueError('Ground-truth clear, cloud, and invalid value sets must be disjoint')
decoder_clear = {0} - declared_invalid
decoder_cloud = {1, 255} - declared_invalid
if declared_clear != decoder_clear or declared_cloud != decoder_cloud:
    raise ValueError(
        'raw_audit encoding must match preprocess_95cloud.decode_ground_truth for this run'
    )
allowed_gt_values = {0, 1, 255, *declared_invalid}

def raw_file_fingerprint(scene_files):
    records = []
    for scene_id, files in sorted(scene_files.items()):
        for band in ('red', 'green', 'blue', 'gt'):
            path = Path(files[band]).resolve()
            stat = path.stat()
            records.append({
                'scene_id': scene_id,
                'band': band,
                'path': str(path.relative_to(DATA_ROOT)),
                'size': int(stat.st_size),
                'mtime_ns': int(stat.st_mtime_ns),
            })
    return canonical_hash({
        'schema_version': 2,
        'dataset_root': str(DATA_ROOT),
        'declared_audit': raw_audit,
        'files': records,
    })

raw_cache_path = DRIVE_AUDIT_CACHE
ensure_drive_directory(raw_cache_path.parent)
quick_fingerprint = raw_file_fingerprint(SCENE_FILES)
cached_raw_manifest = None
if CFG['reuse_raw_audit_cache'] and raw_cache_path.is_file():
    try:
        candidate_cache = json.loads(raw_cache_path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        candidate_cache = None
    if (
        isinstance(candidate_cache, dict)
        and candidate_cache.get('audit_cache_schema_version') == 2
        and candidate_cache.get('quick_fingerprint') == quick_fingerprint
        and candidate_cache.get('declared_audit') == raw_audit
        and candidate_cache.get('scene_count') == len(SCENE_FILES)
    ):
        cached_raw_manifest = candidate_cache
        print('Reusing raw audit cache:', raw_cache_path)
    else:
        print('Raw audit cache is stale or incompatible; running full audit.')

raw_scenes = []
observed_gt_values = set()
audit_scene_files = {} if cached_raw_manifest is not None else SCENE_FILES
for scene_id, files in audit_scene_files.items():
    bands = []
    expected_shape = None
    for band in ('red', 'green', 'blue'):
        path = Path(files[band])
        array = np.asarray(tifffile.imread(path))
        if array.ndim != 2 or array.dtype != np.uint16:
            raise ValueError(f'{scene_id}/{band} must be a uint16 2D TIFF, got {array.dtype} {array.shape}')
        expected_shape = expected_shape or array.shape
        if array.shape != expected_shape:
            raise ValueError(f'RGB shape mismatch in scene {scene_id}')
        bands.append({
            'band': band,
            'path': str(path.relative_to(DATA_ROOT)),
            'sha256': sha256_file(path),
            'dtype': str(array.dtype),
            'shape': [int(value) for value in array.shape],
            'min': int(array.min()),
            'max': int(array.max()),
        })
    gt_path = Path(files['gt'])
    ground_truth = np.asarray(tifffile.imread(gt_path))
    if ground_truth.ndim != 2 or ground_truth.shape != expected_shape:
        raise ValueError(f'Ground truth shape mismatch in scene {scene_id}')
    values = sorted(int(value) for value in np.unique(ground_truth))
    unexpected = sorted(set(values) - allowed_gt_values)
    if unexpected:
        raise ValueError(
            f'Unaudited GT values in {scene_id}: {unexpected}. Update the audit before preprocessing.'
        )
    observed_gt_values.update(values)
    raw_scenes.append({
        'scene_id': scene_id,
        'bands': bands,
        'ground_truth': {
            'path': str(gt_path.relative_to(DATA_ROOT)),
            'sha256': sha256_file(gt_path),
            'dtype': str(ground_truth.dtype),
            'shape': [int(value) for value in ground_truth.shape],
            'values': values,
        },
    })

if cached_raw_manifest is None:
    raw_manifest = {
        'schema_version': 1,
        'audit_cache_schema_version': 2,
        'dataset_root': str(DATA_ROOT),
        'scene_count': len(raw_scenes),
        'observed_ground_truth_values': sorted(observed_gt_values),
        'declared_audit': raw_audit,
        'scenes': raw_scenes,
        'quick_fingerprint': quick_fingerprint,
    }
    raw_manifest['raw_manifest_id'] = canonical_hash(raw_manifest)
    write_checked_text(
        raw_cache_path,
        json.dumps(raw_manifest, indent=2, sort_keys=True),
        replace_stale=True,
    )
else:
    raw_manifest = cached_raw_manifest
raw_manifest_path = RESULTS / 'raw_dataset_audit.json'
raw_manifest_path.write_text(json.dumps(raw_manifest, indent=2, sort_keys=True), encoding='utf-8')

unverified = [
    field for field in ('sensor_id', 'platform_id', 'product_type', 'processing_level', 'units',
                  'scale_offset', 'nodata', 'saturation', 'gsd')
    if raw_audit[field] == 'UNVERIFIED'
]
if CFG['run_mode'] == 'release_candidate' and (unverified or not raw_audit['ground_truth_encoding_confirmed']):
    raise RuntimeError(
        'Release-candidate mode requires a completed raw_audit. Unverified fields: ' + ', '.join(unverified)
    )

print('Raw manifest ID:', raw_manifest['raw_manifest_id'])
print('Observed GT values:', raw_manifest['observed_ground_truth_values'])
if CFG['run_mode'] == 'research_baseline':
    print('Research baseline mode: radiometry and pretrained-artifact release gates remain open.')


## 7. Run the SegFormer regression tests

The targeted suite covers mask/validity semantics, native-size loss, threshold selection, postprocess parity, products, and the fixed graph contract.


In [ ]:
if CFG['run_regression_tests']:
    run_command(
        'SegFormer regression tests',
        [sys.executable, '-m', 'pytest', 'tests/test_segformer_integration.py', '-q'],
        cwd=PROJECT,
    )
else:
    print('Regression tests skipped by configuration.')


## 8. Preprocess native scenes and split at scene level

Preprocessing assigns scenes deterministically before reading them and writes the native RGB image, cloud target, and validity mask directly into local `train/val/test` directories. It does not create an intermediate `all/` tree or hash every generated NPY. Raw TIFFs remain on Drive; the local processed split is archived as one tar file on Drive after a successful build. Set `rebuild_processed_data=True` to force regeneration.


In [ ]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import tarfile
from src.data.preprocess_95cloud import decode_ground_truth
from src.data.split_dataset import split_scenes

split_names = ('train', 'val', 'test')
processed_cache_metadata_path = PROCESSED / 'processed_cache_metadata.json'
expected_processed_cache = {
    'schema_version': 2,
    'cache_name': CFG['processed_cache_name'],
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'raw_quick_fingerprint': raw_manifest.get('quick_fingerprint'),
    'source_revision': source_revision,
    'input_spec_id': segformer_manifest['input_spec']['input_spec_id'],
    'artifacts': {
        'image': True,
        'mask': True,
        'validity': True,
        'raw_masks': bool(CFG['persist_raw_masks']),
    },
    'preprocess': {
        'channels': 3,
        'keep_native_size': True,
        'cloud_ratio_threshold': CFG['cloud_ratio_threshold'],
        'invalid_ground_truth_values': [int(value) for value in raw_audit['invalid_ground_truth_values']],
    },
    'split': {
        'val_ratio': CFG['val_ratio'],
        'test_ratio': CFG['test_ratio'],
        'seed': CFG['seed'],
        'method': 'direct-scene-assignment-v2',
    },
    'storage': {
        'active': 'local',
        'persistent': 'drive-tar',
    },
}

def split_ready(root=PROCESSED):
    manifest = Path(root) / 'scene_split_manifest.json'
    lineage = Path(root) / 'scene_split_lineage.json'
    return manifest.is_file() and lineage.is_file() and all(
        (Path(root) / split / 'masks').is_dir()
        and (Path(root) / split / 'validity').is_dir()
        and any((Path(root) / split / 'masks').glob('*.npy'))
        for split in split_names
    )

def cache_metadata_matches(path):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8')) == expected_processed_cache
    except (OSError, json.JSONDecodeError):
        return False

def processed_cache_ready(root=PROCESSED, metadata_path=None):
    metadata_path = metadata_path or Path(root) / 'processed_cache_metadata.json'
    return CFG['reuse_processed_cache'] and split_ready(root) and cache_metadata_matches(metadata_path)

def estimated_processed_bytes():
    total = 0
    for scene in raw_manifest['scenes']:
        height, width = (int(value) for value in scene['ground_truth']['shape'])
        rgb_itemsize = np.dtype(scene['bands'][0]['dtype']).itemsize
        total += height * width * (3 * rgb_itemsize + 2 * np.dtype(np.uint8).itemsize)
    return int(total * 1.15 + 128 * 1024 ** 2)

def ensure_local_capacity(required_bytes):
    free_bytes = shutil.disk_usage(CONTENT).free
    reserve_bytes = int(CFG['processed_cache_reserve_gib'] * 1024 ** 3)
    if required_bytes + reserve_bytes > free_bytes:
        raise RuntimeError(
            f'Local Colab storage is too small: estimated processed data is '
            f'{required_bytes / 1024 ** 3:.1f} GiB, free={free_bytes / 1024 ** 3:.1f} GiB, '
            f'reserve={CFG["processed_cache_reserve_gib"]:.1f} GiB. '
            'Keep raw TIFFs on Drive and reduce the native dataset or free local storage.'
        )

def clear_local_processed():
    remove_content_path(LOCAL_PROCESSED)
    PROCESSED.mkdir(parents=True, exist_ok=True)

def archive_processed_cache():
    if not CFG['processed_cache_archive']:
        return
    partial = DRIVE_PROCESSED_ARCHIVE.with_name(DRIVE_PROCESSED_ARCHIVE.name + '.partial')
    existing_archive = DRIVE_PROCESSED_ARCHIVE.exists()
    existing_metadata = DRIVE_PROCESSED_METADATA.exists()
    if existing_archive or existing_metadata:
        if (
            existing_archive
            and existing_metadata
            and DRIVE_PROCESSED_ARCHIVE.is_file()
            and DRIVE_PROCESSED_METADATA.is_file()
            and cache_metadata_matches(DRIVE_PROCESSED_METADATA)
        ):
            if partial.exists():
                print('Removing stale partial cache archive:', partial)
                partial.unlink()
            print('Reusing existing processed cache archive:', DRIVE_PROCESSED_ARCHIVE)
            return
        raise FileExistsError(
            f'Processed cache target already exists with different or incomplete data: '
            f'{DRIVE_PROCESSED_ARCHIVE} / {DRIVE_PROCESSED_METADATA}. '
            'Choose a new processed_cache_name instead of overwriting it.'
        )
    if partial.exists():
        print('Removing incomplete processed cache archive:', partial)
        partial.unlink()
    with tarfile.open(partial, 'w') as archive:
        for path in sorted(PROCESSED.rglob('*')):
            if path.is_file():
                archive.add(path, arcname=str(path.relative_to(PROCESSED)).replace('\\', '/'), recursive=False)
    partial.replace(DRIVE_PROCESSED_ARCHIVE)
    save_to_drive(processed_cache_metadata_path, DRIVE_PROCESSED_METADATA)
    print('Persisted processed cache archive:', DRIVE_PROCESSED_ARCHIVE)

def extract_processed_cache():
    clear_local_processed()
    ensure_local_capacity(estimated_processed_bytes())
    root = PROCESSED.resolve()
    with tarfile.open(DRIVE_PROCESSED_ARCHIVE, 'r') as archive:
        members = archive.getmembers()
        for member in members:
            target = (PROCESSED / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe processed cache member: {member.name}')
            if member.issym() or member.islnk():
                raise RuntimeError(f'Links are not allowed in processed cache: {member.name}')
        archive.extractall(PROCESSED)
    print('Extracted processed cache to local storage:', PROCESSED)

def load_native_scene(scene):
    files = SCENE_FILES[scene]
    arrays = [np.asarray(tifffile.imread(files[band])) for band in ('red', 'green', 'blue')]
    ground_truth = np.asarray(tifffile.imread(files['gt']))
    if any(array.ndim != 2 for array in arrays) or ground_truth.ndim != 2:
        raise ValueError(f'Expected 2D RGB channels and GT for scene {scene}')
    if len({array.shape for array in (*arrays, ground_truth)}) != 1:
        raise ValueError(f'Channel/mask shape mismatch for scene {scene}')
    image = np.stack(arrays, axis=-1)
    cloud_mask, validity_mask = decode_ground_truth(
        ground_truth, invalid_values=raw_audit['invalid_ground_truth_values']
    )
    valid_count = int(np.count_nonzero(validity_mask))
    cloud_ratio = float(np.count_nonzero(cloud_mask)) / valid_count if valid_count else 0.0
    label = 'cloud' if cloud_ratio >= CFG['cloud_ratio_threshold'] else 'clear'
    return scene, image, cloud_mask, validity_mask, ground_truth, label

def preprocess_direct_to_splits():
    estimated_bytes = estimated_processed_bytes()
    clear_local_processed()
    ensure_local_capacity(estimated_bytes)
    artifact_dirs = ['cloud', 'clear', 'masks', 'validity']
    if CFG['persist_raw_masks']:
        artifact_dirs.append('raw_masks')
    for split in split_names:
        for directory in artifact_dirs:
            (PROCESSED / split / directory).mkdir(parents=True, exist_ok=True)

    scene_splits = split_scenes(
        SCENE_FILES.keys(), CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
    )
    counts = {split: {'cloud': 0, 'clear': 0} for split in split_names}
    for split in split_names:
        scenes = scene_splits[split]
        with ThreadPoolExecutor(max_workers=max(1, int(CFG['preprocess_workers']))) as executor:
            loaded_scenes = executor.map(load_native_scene, scenes)
            for scene, image, cloud_mask, validity_mask, ground_truth, label in tqdm(
                loaded_scenes, total=len(scenes), desc=f'95-Cloud {split}'
            ):
                filename = f'{scene}_p0.npy'
                np.save(PROCESSED / split / label / filename, image, allow_pickle=False)
                np.save(PROCESSED / split / 'masks' / filename, cloud_mask, allow_pickle=False)
                np.save(PROCESSED / split / 'validity' / filename, validity_mask, allow_pickle=False)
                if CFG['persist_raw_masks']:
                    np.save(PROCESSED / split / 'raw_masks' / filename, ground_truth, allow_pickle=False)
                counts[split][label] += 1

    lineage_payload = {
        'schema_version': 2,
        'raw_manifest_id': raw_manifest['raw_manifest_id'],
        'raw_quick_fingerprint': raw_manifest.get('quick_fingerprint'),
        'scene_splits': {key: sorted(value) for key, value in sorted(scene_splits.items())},
        'preprocess': expected_processed_cache['preprocess'],
    }
    lineage_id = canonical_hash(lineage_payload)
    manifest = {}
    for split in split_names:
        image_count = counts[split]['cloud'] + counts[split]['clear']
        manifest[split] = {
            'scene_count': len(scene_splits[split]),
            'scenes': scene_splits[split],
            'patch_counts': counts[split],
            'image_count': image_count,
            'mask_count': image_count,
            'pairing_valid': True,
            'lineage_id': lineage_id,
            'validity_artifact': True,
            'raw_ground_truth_artifact': bool(CFG['persist_raw_masks']),
        }
    split_lineage = {
        'schema_version': 2,
        'lineage_id': lineage_id,
        'source_directory': str(Path(DATA_ROOT).resolve()),
        'scene_splits': scene_splits,
        'validity_artifact': True,
        'raw_ground_truth_artifact': bool(CFG['persist_raw_masks']),
        'lineage_basis': 'raw-manifest-id-scene-assignment-preprocess-config',
    }
    (PROCESSED / 'scene_split_manifest.json').write_text(
        json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8'
    )
    (PROCESSED / 'scene_split_lineage.json').write_text(
        json.dumps(split_lineage, indent=2, sort_keys=True), encoding='utf-8'
    )
    processed_cache_metadata_path.write_text(
        json.dumps(expected_processed_cache, indent=2, sort_keys=True), encoding='utf-8'
    )
    print(f'Preprocessing completed directly into train/val/test: {len(SCENE_FILES)} scenes')
    print(f'Estimated local processed size: {estimated_bytes / 1024 ** 3:.1f} GiB')
    archive_processed_cache()

local_cache_ready = processed_cache_ready()
drive_cache_ready = (
    CFG['reuse_processed_cache']
    and DRIVE_PROCESSED_ARCHIVE.is_file()
    and cache_metadata_matches(DRIVE_PROCESSED_METADATA)
)
if CFG['rebuild_processed_data']:
    preprocess_direct_to_splits()
elif local_cache_ready:
    print('Using local processed scene split:', PROCESSED)
elif drive_cache_ready:
    try:
        extract_processed_cache()
    except (OSError, tarfile.TarError, RuntimeError) as exc:
        print(f'Drive processed cache could not be extracted: {exc}')
        preprocess_direct_to_splits()
else:
    preprocess_direct_to_splits()

split_manifest_path = PROCESSED / 'scene_split_manifest.json'
split_lineage_path = PROCESSED / 'scene_split_lineage.json'
if not split_ready() or not split_lineage_path.is_file():
    raise RuntimeError('Processed scene split or lineage manifest is incomplete')
split_manifest = json.loads(split_manifest_path.read_text(encoding='utf-8'))
split_lineage = json.loads(split_lineage_path.read_text(encoding='utf-8'))
print('Processed split lineage:', split_lineage['lineage_id'])
disk_report('After preprocessing and split')


## 9. Validate the segmentation dataset and derive train-only statistics

This checks split leakage, target values, validity semantics, tensor finiteness, and native spatial shapes. The train-only RGB statistics are recorded for the data audit. The released MVP `InputSpec` is still pinned to its dtype-range normalization, so these values are not silently applied to this run.


In [ ]:
from src.data.segmentation_dataset import SegmentationDataset

split_names = ('train', 'val', 'test')
validation_cache_context = {
    'source_revision': source_revision,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'processed_cache_name': CFG['processed_cache_name'],
    'preserve_native_size': True,
}
validation_cache_key = canonical_hash(validation_cache_context)[:16]
validation_cache_dir = DRIVE_CACHE / 'validation'
validation_cache_metadata_path = validation_cache_dir / f'dataset_validation_{validation_cache_key}.json'
validation_cache_summary_path = validation_cache_dir / f'dataset_summary_{validation_cache_key}.json'
validation_cache_stats_path = validation_cache_dir / f'train_rgb_statistics_{validation_cache_key}.json'
local_validation_metadata_path = RESULTS / 'dataset_validation_cache.json'
local_summary_path = RESULTS / 'dataset_validation.json'
local_stats_path = RESULTS / 'train_rgb_statistics.json'
validation_cache_ready = (
    local_summary_path.is_file()
    and local_stats_path.is_file()
    and local_validation_metadata_path.is_file()
    and json_cache_matches(local_validation_metadata_path, validation_cache_context)
)
if not validation_cache_ready and validation_cache_metadata_path.is_file() and validation_cache_summary_path.is_file() and validation_cache_stats_path.is_file():
    validation_cache_ready = json_cache_matches(validation_cache_metadata_path, validation_cache_context)
    if validation_cache_ready:
        shutil.copy2(validation_cache_summary_path, local_summary_path)
        shutil.copy2(validation_cache_stats_path, local_stats_path)
        local_validation_metadata_path.write_text(
            json.dumps({'_cube_nano_cache': validation_cache_context}, indent=2, sort_keys=True),
            encoding='utf-8',
        )
        print('Reusing cached dataset validation:', validation_cache_key)
if validation_cache_ready:
    dataset_summary = json.loads(local_summary_path.read_text(encoding='utf-8'))
    train_statistics = json.loads(local_stats_path.read_text(encoding='utf-8'))
else:
    scene_sets = {split: set(split_manifest[split]['scenes']) for split in split_names}
    for index, left in enumerate(split_names):
        for right in split_names[index + 1:]:
            overlap = sorted(scene_sets[left] & scene_sets[right])
            if overlap:
                raise ValueError(f'Scene leakage between {left} and {right}: {overlap[:5]}')

    dataset_summary = {}
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sumsq = np.zeros(3, dtype=np.float64)
    train_valid_pixels = 0
    for split in split_names:
        dataset = SegmentationDataset(PROCESSED / split, is_train=False, preserve_native_size=True)
        if len(dataset) != int(split_manifest[split]['image_count']):
            raise ValueError(f'{split} sample count differs from its split manifest')
        valid_pixels = 0
        cloud_pixels = 0
        source_pixels = 0
        spatial_shapes = set()
        source_scenes = set()
        for sample in dataset:
            image = sample['image']
            target = sample['mask']
            validity = sample['validity_mask']
            if image.dtype != torch.float32 or image.ndim != 3 or image.shape[0] != 3:
                raise TypeError(f'Invalid image tensor in {split}: {tuple(image.shape)} {image.dtype}')
            if not torch.isfinite(image).all() or target.shape != validity.shape or target.shape != image.shape[1:]:
                raise ValueError(f'Invalid image/mask/validity alignment in {split}')
            valid_values = target[validity]
            if valid_values.numel() and not torch.all((valid_values == 0) | (valid_values == 1)):
                raise ValueError(f'Valid target contains values outside 0/1 in {split}')
            if torch.any(target[~validity] != 255):
                raise ValueError(f'Invalid pixels are not encoded as ignore_index in {split}')
            source_pixels += int(target.numel())
            valid_pixels += int(validity.sum())
            cloud_pixels += int((target[validity] == 1).sum())
            spatial_shapes.add(tuple(int(value) for value in target.shape))
            source_scenes.add(sample['scene_id'])
            if split == 'train' and validity.any():
                values = image[:, validity].double().cpu().numpy()
                channel_sum += values.sum(axis=1)
                channel_sumsq += np.square(values).sum(axis=1)
                train_valid_pixels += values.shape[1]
        dataset_summary[split] = {
            'samples': len(dataset),
            'scene_count': len(source_scenes),
            'source_pixels': source_pixels,
            'valid_pixels': valid_pixels,
            'valid_pixel_ratio': valid_pixels / source_pixels if source_pixels else 0.0,
            'cloud_pixel_ratio_on_valid': cloud_pixels / valid_pixels if valid_pixels else None,
            'native_spatial_shapes': [list(shape) for shape in sorted(spatial_shapes)],
        }

    if train_valid_pixels <= 0:
        raise RuntimeError('Train split contains no valid pixels')
    train_mean = channel_sum / train_valid_pixels
    train_variance = np.maximum(channel_sumsq / train_valid_pixels - np.square(train_mean), 0.0)
    train_statistics = {
        'dataset_role': 'train',
        'valid_pixel_count': int(train_valid_pixels),
        'tensor_space': 'uint16 divided by 65535',
        'band_order': ['red', 'green', 'blue'],
        'mean': [float(value) for value in train_mean],
        'std': [float(value) for value in np.sqrt(train_variance)],
        'applied_to_this_run': False,
        'reason': 'The pinned MVP InputSpec permits dtype-range normalization only. A mean/std ablation requires a versioned InputSpec and runtime parity update.',
    }
    local_summary_path.write_text(json.dumps(dataset_summary, indent=2, sort_keys=True), encoding='utf-8')
    local_stats_path.write_text(json.dumps(train_statistics, indent=2, sort_keys=True), encoding='utf-8')
    local_validation_metadata_path.write_text(
        json.dumps({'_cube_nano_cache': validation_cache_context}, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    save_to_drive(local_summary_path, validation_cache_summary_path)
    save_to_drive(local_stats_path, validation_cache_stats_path)
    save_to_drive(local_validation_metadata_path, validation_cache_metadata_path)
print(json.dumps(dataset_summary, indent=2, sort_keys=True))
print(json.dumps(train_statistics, indent=2, sort_keys=True))


## 10. Train SegFormer-B0

The repository entry point uses masked Cross-Entropy plus masked Soft Dice, native labels, bilinear logit upsampling, AMP on CUDA, and skips all-invalid batches. Set `CFG['train_batch_size']`, `CFG['train_preserve_native_size']`, and `CFG['use_pretrained_segformer']` in cell 4 to change the training mode; no source-code edit is needed for later runs. When enabled, the notebook downloads/caches `nvidia/mit-b0`, loads its MiT-B0 encoder, and initializes the two-class cloud decoder from scratch. Native-size batches are padded dynamically and padding is excluded from the loss. This cell also logs epoch history and the selected checkpoint to Weights & Biases. It currently selects its checkpoint by validation loss, which is recorded in the bundle; a release candidate still needs an owner-approved checkpoint-selection metric.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training SegFormer-B0 in Colab')

import src.train_segmentation as segmentation_training_module
from src.train_segmentation import SegmentationTrainingConfig, train
import importlib.util
import subprocess
import time
from dataclasses import asdict
from tqdm.auto import tqdm

checkpoint_path = CHECKPOINTS / f"segformer_b0_rgb_{CFG['run_mode']}.pth"
drive_checkpoint_candidate = DRIVE_CHECKPOINTS / checkpoint_path.name
drive_training_report_candidate = DRIVE_RESULTS / checkpoint_path.with_suffix('.json').name
expected_training_config = {
    'epochs': int(CFG['epochs']),
    'learning_rate': float(CFG['learning_rate']),
    'weight_decay': float(CFG['weight_decay']),
    'warmup_epochs': int(CFG['warmup_epochs']),
    'early_stopping_patience': int(CFG['early_stopping_patience']),
    'cross_entropy_weight': 1.0,
    'dice_weight': 1.0,
    'dice_epsilon': 1e-6,
    'ignore_index': 255,
    'seed': int(CFG['seed']),
    'use_amp': bool(CFG['use_amp']),
    'batch_size': int(CFG['train_batch_size']),
    'preserve_native_size': bool(CFG['train_preserve_native_size']),
    'pretrained_encoder_path': str(DRIVE_CACHE / 'pretrained' / 'mit_b0_encoder_hf.pth')
        if CFG['use_pretrained_segformer'] else None,
}
def find_reusable_training_report():
    candidates = (
        (checkpoint_path, checkpoint_path.with_suffix('.json')),
        (drive_checkpoint_candidate, drive_training_report_candidate),
    )
    for candidate_checkpoint, candidate_report_path in candidates:
        if not candidate_checkpoint.is_file() or not candidate_report_path.is_file():
            continue
        if candidate_checkpoint != checkpoint_path and (
            checkpoint_path.exists() or checkpoint_path.with_suffix('.json').exists()
        ):
            continue
        try:
            candidate_report = json.loads(candidate_report_path.read_text(encoding='utf-8'))
            candidate_metadata = candidate_report.get('metadata', {})
            pretrained_metadata = candidate_metadata.get('pretrained_artifact', {})
            if candidate_report.get('training_config') != expected_training_config:
                continue
            if any(candidate_metadata.get(key) != expected for key, expected in {
                'source_revision': source_revision,
                'raw_manifest_id': raw_manifest['raw_manifest_id'],
                'split_lineage_id': split_lineage['lineage_id'],
                'input_spec_id': segformer_manifest['input_spec']['input_spec_id'],
            }.items()):
                continue
            if CFG['use_pretrained_segformer'] and (
                pretrained_metadata.get('source_model_id') != CFG['pretrained_segformer_model_id']
                or pretrained_metadata.get('checkpoint_sha256') != sha256_file(
                    Path(expected_training_config['pretrained_encoder_path'])
                )
            ):
                continue
            checkpoint = torch.load(candidate_checkpoint, map_location='cpu')
            if not isinstance(checkpoint, dict) or 'model_state_dict' not in checkpoint:
                continue
            if candidate_checkpoint != checkpoint_path:
                copy_if_absent_or_same(candidate_checkpoint, checkpoint_path)
                copy_if_absent_or_same(candidate_report_path, checkpoint_path.with_suffix('.json'))
            return candidate_report
        except (OSError, RuntimeError, ValueError, json.JSONDecodeError):
            continue
    return None
reused_training_report = find_reusable_training_report()
if reused_training_report is not None:
    print('Reusing existing compatible SegFormer checkpoint:', checkpoint_path)

WANDB_ENABLED = reused_training_report is None
WANDB_PROJECT = 'cube-nano'
WANDB_ENTITY = None
WANDB_RUN_NAME = f"segformer-b0-{CFG['run_mode']}-seed-{CFG['seed']}"
WANDB_GROUP = '95-cloud-segformer'
WANDB_TAGS = ['colab', '95-cloud', 'segformer-b0', 'segmentation']
WANDB_MODE = 'online'  # Set to 'offline' when network/API access is unavailable.
WANDB_LOG_CHECKPOINT_ARTIFACT = True

if WANDB_ENABLED:
    if importlib.util.find_spec('wandb') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'])
    if WANDB_MODE == 'online' and not os.environ.get('WANDB_API_KEY'):
        try:
            from google.colab import userdata
            wandb_secret = userdata.get('WANDB_API_KEY')
        except Exception:
            wandb_secret = None
        if wandb_secret:
            os.environ['WANDB_API_KEY'] = wandb_secret
    if WANDB_MODE == 'online' and not os.environ.get('WANDB_API_KEY'):
        raise RuntimeError(
            'W&B online logging requires WANDB_API_KEY. Add it to Colab Secrets '
            'or set WANDB_MODE=\'offline\'.'
        )
    import wandb

pretrained_encoder_path = None
pretrained_encoder_sha256 = None
if CFG['use_pretrained_segformer']:
    from transformers import SegformerModel

    pretrained_cache_dir = DRIVE_CACHE / 'pretrained'
    pretrained_cache_dir.mkdir(parents=True, exist_ok=True)
    pretrained_encoder_path = pretrained_cache_dir / 'mit_b0_encoder_hf.pth'
    pretrained_encoder_metadata_path = pretrained_cache_dir / 'mit_b0_encoder_hf.json'
    pretrained_cache_ready = False
    if pretrained_encoder_path.is_file() and pretrained_encoder_metadata_path.is_file():
        try:
            pretrained_metadata = json.loads(
                pretrained_encoder_metadata_path.read_text(encoding='utf-8')
            )
            pretrained_cache_ready = (
                pretrained_metadata.get('source_model_id') == CFG['pretrained_segformer_model_id']
                and pretrained_metadata.get('sha256') == sha256_file(pretrained_encoder_path)
            )
        except (OSError, json.JSONDecodeError):
            pretrained_cache_ready = False
    if pretrained_cache_ready:
        print('Reusing cached SegFormer pretrained encoder on Drive:', pretrained_encoder_path)
    else:
        print(
            'Downloading SegFormer pretrained encoder:',
            CFG['pretrained_segformer_model_id'],
        )
        hf_cache_dir = DRIVE_CACHE / 'huggingface'
        hf_cache_dir.mkdir(parents=True, exist_ok=True)
        hf_encoder = SegformerModel.from_pretrained(
            CFG['pretrained_segformer_model_id'],
            cache_dir=str(hf_cache_dir),
        )
        torch.save(
            {
                'state_dict': hf_encoder.state_dict(),
                'source_model_id': CFG['pretrained_segformer_model_id'],
            },
            pretrained_encoder_path,
        )
        del hf_encoder
        pretrained_encoder_sha256 = sha256_file(pretrained_encoder_path)
        write_checked_text(
            pretrained_encoder_metadata_path,
            json.dumps({
                'source_model_id': CFG['pretrained_segformer_model_id'],
                'sha256': pretrained_encoder_sha256,
            }, indent=2, sort_keys=True),
            replace_stale=True,
        )
        print('Saved pretrained encoder cache on Drive:', pretrained_encoder_path)
    pretrained_encoder_sha256 = pretrained_encoder_sha256 or sha256_file(pretrained_encoder_path)
    print('Pretrained encoder SHA-256:', pretrained_encoder_sha256)

TRAIN_BATCH_SIZE = int(CFG['train_batch_size'])
TRAIN_PRESERVE_NATIVE_SIZE = bool(CFG['train_preserve_native_size'])
training_config = SegmentationTrainingConfig(
    epochs=CFG['epochs'],
    learning_rate=CFG['learning_rate'],
    weight_decay=CFG['weight_decay'],
    warmup_epochs=CFG['warmup_epochs'],
    early_stopping_patience=CFG['early_stopping_patience'],
    cross_entropy_weight=1.0,
    dice_weight=1.0,
    dice_epsilon=1e-6,
    ignore_index=255,
    seed=CFG['seed'],
    use_amp=CFG['use_amp'],
    batch_size=TRAIN_BATCH_SIZE,
    preserve_native_size=TRAIN_PRESERVE_NATIVE_SIZE,
    pretrained_encoder_path=pretrained_encoder_path,
)
print(
    f"Training input mode: {'native scene size' if TRAIN_PRESERVE_NATIVE_SIZE else 'random 256x256 crop'} | "
    f"validation input mode: {'native scene size' if TRAIN_PRESERVE_NATIVE_SIZE else 'deterministic padded 256x256 tiles'} | "
    f"batch_size={TRAIN_BATCH_SIZE}"
)
checkpoint_metadata = {
    'run_mode': CFG['run_mode'],
    'model_task': segformer_manifest['model_task'],
    'model_release_id': segformer_manifest['model_release_id'],
    'source_revision': source_revision,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'input_spec_id': segformer_manifest['input_spec']['input_spec_id'],
    'checkpoint_selection_metric': 'validation_loss',
    'seed_count': 1,
    'seed_limitation': 'This notebook runs one seed. The evaluation bootstrap quantifies scene sampling uncertainty but does not replace a multi-seed final candidate run.',
    'class_mapping': {'clear': 0, 'cloud': 1},
    'pretrained_artifact': {
        'artifact_id': segformer_manifest['implementation']['pretrained_artifact_id'],
        'load_status': 'loaded' if pretrained_encoder_path else 'not_loaded',
        'source_model_id': CFG['pretrained_segformer_model_id'] if pretrained_encoder_path else None,
        'checkpoint_path': str(pretrained_encoder_path) if pretrained_encoder_path else None,
        'checkpoint_sha256': pretrained_encoder_sha256,
        'loaded_component': 'MiT-B0 encoder; cloud segmentation decoder initialized from scratch',
    },
}
wandb_run = None
wandb_metadata = {'enabled': False}
if WANDB_ENABLED:
    wandb_config = {
        'training': asdict(training_config),
        'run_mode': CFG['run_mode'],
        'source_revision': source_revision,
        'raw_manifest_id': raw_manifest['raw_manifest_id'],
        'split_lineage_id': split_lineage['lineage_id'],
        'input_spec_id': segformer_manifest['input_spec']['input_spec_id'],
        'dataset': {
            'cache_name': CFG['processed_cache_name'],
            'split_counts': {
                split: int(split_manifest[split]['image_count']) for split in ('train', 'val', 'test')
            },
            'train_valid_pixel_count': int(train_statistics['valid_pixel_count']),
        },
    }
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        tags=WANDB_TAGS,
        mode=WANDB_MODE,
        config=wandb_config,
    )
    wandb_metadata = {
        'enabled': True,
        'mode': WANDB_MODE,
        'project': WANDB_PROJECT,
        'entity': WANDB_ENTITY,
        'run_id': wandb_run.id,
        'run_name': wandb_run.name,
        'url': wandb_run.url,
    }
    checkpoint_metadata['wandb'] = wandb_metadata

live_epoch_state = {'epoch': 0, 'train_metrics': None, 'learning_rate': None, 'wandb_error': None}
original_train_one_epoch = segmentation_training_module.train_one_epoch
original_evaluate_loss = segmentation_training_module.evaluate_loss

def live_train_one_epoch(*args, **kwargs):
    optimizer = args[2] if len(args) > 2 else kwargs.get('optimizer')
    if optimizer is not None and optimizer.param_groups:
        live_epoch_state['learning_rate'] = float(optimizer.param_groups[0]['lr'])
    loader = args[1] if len(args) > 1 else kwargs['loader']
    train_config = args[4] if len(args) > 4 else kwargs['config']
    epoch = int(live_epoch_state['epoch']) + 1
    started_at = time.perf_counter()
    iteration_count = 0
    with tqdm(
        total=len(loader),
        desc=f'Train {epoch:03d}/{train_config.epochs}',
        unit='it',
        leave=True,
    ) as progress:
        def progress_batches():
            nonlocal iteration_count
            for batch in loader:
                yield batch
                iteration_count += 1
                progress.update(1)
                elapsed = max(time.perf_counter() - started_at, 1e-9)
                progress.set_postfix(
                    lr=f"{live_epoch_state['learning_rate']:.2e}"
                    if live_epoch_state['learning_rate'] is not None else 'n/a',
                    it_s=f"{iteration_count / elapsed:.2f}",
                )
        wrapped_args = list(args)
        if len(wrapped_args) > 1:
            wrapped_args[1] = progress_batches()
            metrics = original_train_one_epoch(*wrapped_args, **kwargs)
        else:
            wrapped_kwargs = dict(kwargs)
            wrapped_kwargs['loader'] = progress_batches()
            metrics = original_train_one_epoch(**wrapped_kwargs)
    elapsed = max(time.perf_counter() - started_at, 1e-9)
    metrics['iterations_per_second'] = iteration_count / elapsed
    live_epoch_state['train_metrics'] = metrics
    live_epoch_state['iterations_per_second'] = metrics['iterations_per_second']
    return metrics

def live_evaluate_loss(model, loader, device, config):
    epoch = int(live_epoch_state['epoch']) + 1
    validation_started_at = time.perf_counter()
    validation_iteration_count = 0
    with tqdm(
        total=len(loader),
        desc=f'Val   {epoch:03d}/{config.epochs}',
        unit='it',
        leave=True,
    ) as progress:
        def progress_batches():
            nonlocal validation_iteration_count
            for batch in loader:
                yield batch
                validation_iteration_count += 1
                progress.update(1)
                elapsed = max(time.perf_counter() - validation_started_at, 1e-9)
                progress.set_postfix(it_s=f"{validation_iteration_count / elapsed:.2f}")
        metrics = original_evaluate_loss(model, progress_batches(), device, config)
    metrics['iterations_per_second'] = validation_iteration_count / max(
        time.perf_counter() - validation_started_at, 1e-9
    )
    train_metrics = live_epoch_state['train_metrics'] or {}
    learning_rate = live_epoch_state['learning_rate']
    live_metrics = {
        'epoch': epoch,
        'train/batch_size': int(config.batch_size),
        'train/learning_rate': float(learning_rate if learning_rate is not None else config.learning_rate),
        'train/iterations_per_second': float(train_metrics.get('iterations_per_second', 0.0)),
        'train/loss': float(train_metrics.get('loss', float('nan'))),
        'validation/loss': float(metrics.get('loss', float('nan'))),
        'train/valid_pixels': int(train_metrics.get('valid_pixels', 0)),
        'validation/valid_pixels': int(metrics.get('valid_pixels', 0)),
        'train/optimizer_steps': int(train_metrics.get('optimizer_steps', 0)),
        'train/skipped_all_invalid_batches': int(train_metrics.get('skipped_all_invalid_batches', 0)),
        'validation/skipped_all_invalid_batches': int(metrics.get('skipped_all_invalid_batches', 0)),
    }
    print(
        f"Epoch {epoch:03d}/{config.epochs} | "
        f"batch_size={live_metrics['train/batch_size']} | "
        f"lr={live_metrics['train/learning_rate']:.2e} | "
        f"train_loss={live_metrics['train/loss']:.6f} | "
        f"val_loss={live_metrics['validation/loss']:.6f} | "
        f"train_valid={live_metrics['train/valid_pixels']:,} | "
        f"val_valid={live_metrics['validation/valid_pixels']:,} | "
        f"optimizer_steps={live_metrics['train/optimizer_steps']}",
        flush=True,
    )
    if wandb_run is not None:
        try:
            wandb_run.log(live_metrics, step=epoch)
        except Exception as exc:
            live_epoch_state['wandb_error'] = repr(exc)
            print('W&B live epoch log failed; continuing training:', exc, flush=True)
    live_epoch_state['epoch'] = epoch
    return metrics

segmentation_training_module.train_one_epoch = live_train_one_epoch
segmentation_training_module.evaluate_loss = live_evaluate_loss
try:
    if reused_training_report is not None:
        training_report = reused_training_report
    else:
        training_report = train(
            PROCESSED / 'train',
            PROCESSED / 'val',
            checkpoint_path,
            config=training_config,
            device='cuda',
            metadata=checkpoint_metadata,
        )
except Exception:
    if wandb_run is not None:
        wandb_run.finish(exit_code=1)
    raise
finally:
    segmentation_training_module.train_one_epoch = original_train_one_epoch
    segmentation_training_module.evaluate_loss = original_evaluate_loss
if not checkpoint_path.is_file():
    raise RuntimeError('Training completed without writing a checkpoint')
checkpoint_sha256 = sha256_file(checkpoint_path)
if wandb_run is not None:
    try:
        best_epoch = min(
            training_report['history'],
            key=lambda record: float(record['validation']['loss']),
        )['epoch']
        wandb_run.summary.update({
            'best_validation_loss': float(training_report['best_validation_loss']),
            'best_epoch': int(best_epoch) + 1,
            'checkpoint_sha256': checkpoint_sha256,
            'checkpoint_path': str(checkpoint_path),
        })
        if WANDB_LOG_CHECKPOINT_ARTIFACT:
            model_artifact = wandb.Artifact(
                f"segformer-b0-{CFG['run_mode']}",
                type='model',
                metadata={
                    'checkpoint_sha256': checkpoint_sha256,
                    'source_revision': source_revision,
                    'split_lineage_id': split_lineage['lineage_id'],
                },
            )
            model_artifact.add_file(str(checkpoint_path), name=checkpoint_path.name)
            model_artifact.add_file(
                str(checkpoint_path.with_suffix('.json')),
                name=checkpoint_path.with_suffix('.json').name,
            )
            wandb_run.log_artifact(model_artifact)
    except Exception as exc:
        wandb_metadata['logging_error'] = repr(exc)
        print('W&B logging failed after training; continuing with local artifacts:', exc)
    finally:
        wandb_run.finish()
drive_checkpoint_path = save_to_drive(
    checkpoint_path, DRIVE_CHECKPOINTS / checkpoint_path.name
)
drive_training_report_path = save_to_drive(
    checkpoint_path.with_suffix('.json'),
    DRIVE_RESULTS / checkpoint_path.with_suffix('.json').name,
)
(RESULTS / 'training_summary.json').write_text(
    json.dumps({
        **training_report,
        'checkpoint_sha256': checkpoint_sha256,
        'drive_checkpoint_path': str(drive_checkpoint_path),
        'drive_training_report_path': str(drive_training_report_path),
        'wandb': wandb_metadata,
    }, indent=2, sort_keys=True),
    encoding='utf-8',
)
print('Checkpoint:', checkpoint_path)
print('Checkpoint SHA-256:', checkpoint_sha256)
print('Checkpoint copied to Drive:', drive_checkpoint_path)
print('Best validation loss:', training_report['best_validation_loss'])


## 11. Calibrate the pixel threshold on validation only

The evaluator sweeps candidate thresholds against validation predictions, maximizes Dice under the false-clear constraint, and writes a candidate `DecisionSpec`. The test split is not used by this cell.


In [ ]:
calibration_path = RESULTS / 'validation_calibration.json'
calibration_cache = {
    'checkpoint_sha256': checkpoint_sha256,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'max_false_clear_rate': CFG['max_false_clear_rate'],
    'threshold_start_bp': CFG['threshold_start_bp'],
    'threshold_stop_bp': CFG['threshold_stop_bp'],
    'threshold_step_bp': CFG['threshold_step_bp'],
    'bootstrap_samples': CFG['bootstrap_samples'],
    'bootstrap_seed': CFG['seed'],
}
if not json_cache_matches(calibration_path, calibration_cache):
    restore_drive_artifact(
        calibration_path,
        DRIVE_RESULTS / calibration_path.name,
        DRIVE_DELIVERABLES / calibration_path.name,
        validator=lambda path: json_cache_matches(path, calibration_cache),
    )
if not json_cache_matches(calibration_path, calibration_cache):
    run_command(
        'Calibrate pixel threshold on validation',
        [
            sys.executable, PROJECT / 'src/eval_segmentation.py',
            '--test-dir', PROCESSED / 'val',
            '--model-path', checkpoint_path,
            '--dataset-role', 'validation',
            '--select-threshold',
            '--max-false-clear-rate', str(CFG['max_false_clear_rate']),
            '--threshold-start-bp', str(CFG['threshold_start_bp']),
            '--threshold-stop-bp', str(CFG['threshold_stop_bp']),
            '--threshold-step-bp', str(CFG['threshold_step_bp']),
            '--bootstrap-samples', str(CFG['bootstrap_samples']),
            '--bootstrap-seed', str(CFG['seed']),
            '--device', 'cuda',
            '--output', calibration_path,
        ],
        cwd=RUN,
    )
calibration_report = json.loads(calibration_path.read_text(encoding='utf-8'))
if calibration_report.get('_cube_nano_cache') != calibration_cache:
    calibration_report['_cube_nano_cache'] = calibration_cache
    calibration_path.write_text(json.dumps(calibration_report, indent=2, sort_keys=True), encoding='utf-8')
locked_threshold_bp = int(calibration_report['threshold_bp'])
candidate_decision_spec = dict(segformer_manifest['decision_spec'])
candidate_decision_spec.update({
    'pixel_cloud_probability_threshold_bp': locked_threshold_bp,
    'false_clear_constraint_bp': int(round(CFG['max_false_clear_rate'] * 10000)),
    'calibration_id': f'95cloud-validation-{checkpoint_sha256[:12]}',
    'calibration_report_sha256': sha256_file(calibration_path),
    'dataset_role': 'validation',
    'split_lineage_id': split_lineage['lineage_id'],
})
locked_decision_path = CONTRACTS / 'candidate_decision_spec.json'
locked_decision_path.write_text(
    json.dumps(candidate_decision_spec, indent=2, sort_keys=True), encoding='utf-8'
)
print('Locked validation threshold (bp):', locked_threshold_bp)
print('Candidate decision spec:', locked_decision_path)


## 12. Run the frozen test evaluation once

This uses the locked validation threshold without a test sweep. It reports micro/macro scene metrics, boundary F1, coverage errors, and deterministic scene bootstrap intervals.


In [ ]:
test_report_path = RESULTS / 'test_evaluation.json'
test_cache = {
    'checkpoint_sha256': checkpoint_sha256,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'threshold_bp': locked_threshold_bp,
    'bootstrap_samples': CFG['bootstrap_samples'],
    'bootstrap_seed': CFG['seed'],
}
if not json_cache_matches(test_report_path, test_cache):
    restore_drive_artifact(
        test_report_path,
        DRIVE_RESULTS / test_report_path.name,
        DRIVE_DELIVERABLES / test_report_path.name,
        validator=lambda path: json_cache_matches(path, test_cache),
    )
if not json_cache_matches(test_report_path, test_cache):
    run_command(
        'Evaluate frozen test split',
        [
            sys.executable, PROJECT / 'src/eval_segmentation.py',
            '--test-dir', PROCESSED / 'test',
            '--model-path', checkpoint_path,
            '--dataset-role', 'test',
            '--threshold-bp', str(locked_threshold_bp),
            '--bootstrap-samples', str(CFG['bootstrap_samples']),
            '--bootstrap-seed', str(CFG['seed']),
            '--device', 'cuda',
            '--output', test_report_path,
        ],
        cwd=RUN,
    )
test_report = json.loads(test_report_path.read_text(encoding='utf-8'))
if test_report.get('_cube_nano_cache') != test_cache:
    test_report['_cube_nano_cache'] = test_cache
    test_report_path.write_text(json.dumps(test_report, indent=2, sort_keys=True), encoding='utf-8')
metrics = test_report['metrics']
coverage = test_report['coverage_metrics']
quality = acceptance_profile['quality']
def check_at_least(name, actual, expected):
    return {'actual': actual, 'expected': expected, 'passed': actual is not None and actual >= expected}

def check_at_most(name, actual, expected):
    return {'actual': actual, 'expected': expected, 'passed': actual is not None and actual <= expected}

quality_gates = {
    'cloud_iou': check_at_least('cloud_iou', metrics['cloud_iou'], quality['min_cloud_iou']),
    'cloud_dice': check_at_least('cloud_dice', metrics['cloud_dice'], quality['min_cloud_dice']),
    'cloud_recall': check_at_least('cloud_recall', metrics['cloud_recall'], quality['min_cloud_recall']),
    'false_clear_rate': check_at_most('false_clear_rate', metrics['false_clear_rate'], quality['max_false_clear_rate']),
    'boundary_f1': check_at_least('boundary_f1', metrics['boundary_f1'], quality['min_boundary_f1']),
    'coverage_mae_bp': check_at_most('coverage_mae_bp', coverage['coverage_mae_bp'], quality['max_coverage_mae_bp']),
    'coverage_p95_abs_error_bp': check_at_most(
        'coverage_p95_abs_error_bp', coverage['coverage_p95_abs_error_bp'], quality['max_coverage_p95_abs_error_bp']
    ),
    'valid_pixel_ratio': check_at_least(
        'valid_pixel_ratio', test_report['valid_pixel_ratio'], acceptance_profile['runtime']['min_valid_pixel_ratio']
    ),
}
quality_gate_report = {
    'run_mode': CFG['run_mode'],
    'checkpoint_sha256': checkpoint_sha256,
    'split_lineage_id': split_lineage['lineage_id'],
    'threshold_bp': locked_threshold_bp,
    'all_quality_gates_passed': all(item['passed'] for item in quality_gates.values()),
    'gates': quality_gates,
    'release_status': 'not_a_release',
    'release_blockers': [
        'This notebook does not create a pinned pretrained artifact.',
        'This notebook runs one training seed; a final candidate requires three seeds or a documented resource exception.',
        'The model manifest, calibration binding, target benchmark, and TensorRT parity are not mutated by this run.',
        'Release promotion remains subject to the integration-plan gates.',
    ],
}
quality_gate_path = RESULTS / 'quality_gate_report.json'
quality_gate_path.write_text(
    json.dumps(quality_gate_report, indent=2, sort_keys=True), encoding='utf-8'
)
print(json.dumps(quality_gate_report, indent=2, sort_keys=True))


## 13. Export the fixed runtime graph and check PyTorch/ONNX parity

The runtime graph stays fixed at batch 1 and `256 x 256`. A padded/cropped normalized validation tile is stored as a golden input together with raw PyTorch and ONNX logits. TensorRT parity must still run on the pinned target outside Colab.


In [ ]:
onnx_path = ARTIFACTS / f"segformer_b0_rgb_{CFG['run_mode']}.onnx"
onnx_contract_path = onnx_path.with_suffix('.onnx.json')
def onnx_cache_matches(path):
    try:
        contract = json.loads(Path(path).read_text(encoding='utf-8'))
        return contract.get('checkpoint_sha256') == checkpoint_sha256
    except (OSError, json.JSONDecodeError):
        return False
onnx_ready = onnx_path.is_file() and onnx_contract_path.is_file() and onnx_cache_matches(onnx_contract_path)
if not onnx_ready:
    restore_drive_artifact(
        onnx_contract_path,
        DRIVE_RESULTS / onnx_contract_path.name,
        DRIVE_DELIVERABLES / onnx_contract_path.name,
        validator=onnx_cache_matches,
    )
    if onnx_contract_path.is_file() and onnx_cache_matches(onnx_contract_path):
        restore_drive_artifact(
            onnx_path,
            DRIVE_RESULTS / onnx_path.name,
            DRIVE_DELIVERABLES / onnx_path.name,
        )
    onnx_ready = onnx_path.is_file() and onnx_contract_path.is_file() and onnx_cache_matches(onnx_contract_path)
if not onnx_ready:
    run_command(
        'Export fixed-shape ONNX',
        [
            sys.executable, PROJECT / 'src/export_segformer_onnx.py',
            '--checkpoint', checkpoint_path,
            '--output', onnx_path,
        ],
        cwd=RUN,
    )

import onnxruntime as ort
from src.models.segformer_b0 import get_segformer_b0

golden_dataset = SegmentationDataset(PROCESSED / 'val', is_train=False, preserve_native_size=True)
golden_sample = golden_dataset[0]['image'].cpu().numpy()
golden_input = np.zeros((1, 3, 256, 256), dtype=np.float32)
copy_height = min(256, golden_sample.shape[1])
copy_width = min(256, golden_sample.shape[2])
golden_input[0, :, :copy_height, :copy_width] = golden_sample[:, :copy_height, :copy_width]

model = get_segformer_b0().eval()
checkpoint = torch.load(checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
with torch.inference_mode():
    pytorch_logits = model(torch.from_numpy(golden_input)).cpu().numpy()
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
onnx_logits = session.run(['logits'], {'input': golden_input})[0]
if pytorch_logits.shape != (1, 2, 64, 64) or onnx_logits.shape != (1, 2, 64, 64):
    raise ValueError(f'Unexpected fixed-shape logits: {pytorch_logits.shape}, {onnx_logits.shape}')
difference = np.abs(pytorch_logits.astype(np.float32) - onnx_logits.astype(np.float32))
onnx_tolerance = float(acceptance_profile['parity']['pytorch_onnx'])
parity_report = {
    'input_shape': list(golden_input.shape),
    'output_shape': list(pytorch_logits.shape),
    'max_abs_difference': float(difference.max()),
    'mean_abs_difference': float(difference.mean()),
    'tolerance': onnx_tolerance,
    'passed': bool(np.allclose(pytorch_logits, onnx_logits, rtol=0.0, atol=onnx_tolerance)),
    'tensorrt': {
        'status': 'not_run',
        'reason': 'TensorRT parity must run on the pinned deployment target, not on a generic Colab GPU.',
    },
}
np.save(ARTIFACTS / 'golden_input.npy', golden_input)
np.save(ARTIFACTS / 'golden_pytorch_logits.npy', pytorch_logits)
np.save(ARTIFACTS / 'golden_onnx_logits.npy', onnx_logits)
parity_path = RESULTS / 'pytorch_onnx_parity.json'
parity_path.write_text(json.dumps(parity_report, indent=2, sort_keys=True), encoding='utf-8')
if not parity_report['passed']:
    raise AssertionError(f'PyTorch/ONNX parity failed: {parity_report}')
print(json.dumps(parity_report, indent=2, sort_keys=True))


## 14. Package the evidence bundle

Only model/reports/contracts/golden vectors are zipped. The raw and processed datasets stay out of the download archive.


In [ ]:
environment_report = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'packages': {
        name: importlib.metadata.version(name)
        for name in ('numpy', 'torch', 'tifffile', 'onnx', 'onnxruntime', 'PyYAML', 'kaggle')
    },
}
if torch.cuda.is_available():
    environment_report['gpu_name'] = torch.cuda.get_device_name(0)
    try:
        environment_report['nvidia_smi'] = subprocess.check_output(['nvidia-smi', '-q'], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        environment_report['nvidia_smi'] = 'unavailable'
environment_path = RESULTS / 'environment.json'
environment_path.write_text(json.dumps(environment_report, indent=2, sort_keys=True), encoding='utf-8')

bundle_files = [
    checkpoint_path,
    checkpoint_path.with_suffix('.json'),
    RESULTS / 'source_provenance.json',
    RESULTS / 'raw_dataset_audit.json',
    RESULTS / 'dataset_validation.json',
    RESULTS / 'train_rgb_statistics.json',
    RESULTS / 'training_summary.json',
    calibration_path,
    test_report_path,
    quality_gate_path,
    parity_path,
    environment_path,
    locked_decision_path,
    split_manifest_path,
    split_lineage_path,
    onnx_path,
    onnx_path.with_suffix('.onnx.json'),
    ARTIFACTS / 'golden_input.npy',
    ARTIFACTS / 'golden_pytorch_logits.npy',
    ARTIFACTS / 'golden_onnx_logits.npy',
]
for source in bundle_files:
    source = Path(source)
    if not source.is_file():
        raise FileNotFoundError(f'Expected artifact is missing: {source}')
    copy_if_absent_or_same(source, DELIVERABLES / source.name)

archive_base = CONTENT / 'segformer_95cloud_evidence_bundle'
archive_target = archive_base.with_suffix('.zip')
drive_archive_candidate = DRIVE_ROOT / archive_target.name
if archive_target.is_file():
    archive_path = archive_target
    print('Reusing existing local evidence bundle:', archive_path)
elif drive_archive_candidate.is_file():
    archive_path = copy_if_absent_or_same(drive_archive_candidate, archive_target)
    print('Reusing existing Drive evidence bundle:', archive_path)
else:
    archive_path = shutil.make_archive(str(archive_base), 'zip', RUN, 'deliverables')
for source in DELIVERABLES.iterdir():
    if source.is_file():
        save_to_drive(source, DRIVE_DELIVERABLES / source.name)
drive_archive_path = save_to_drive(archive_path, DRIVE_ROOT / Path(archive_path).name)
print('Evidence bundle:', archive_path)
print('Evidence bundle copied to Drive:', drive_archive_path)

from google.colab import files
files.download(archive_path)

if CFG['cleanup_local_after_bundle']:
    remove_content_path(LOCAL_PROCESSED)
    disk_report('After local processed-data cleanup')
